# Multi-News Summarization

Generates LLM summaries from the Multi-News dataset (Fabbri et al., 2019)

**Multi-News:**
- Professionally written multi-document digests, averaging ~264 words (~10-13 sentences)    
—> long enough for all metrics to be meaningfully defined, without changing their thresholds
- One human reference summary per article (`summary` field)   
-> only the 5 model generations per article are used for diversity comparisons
-> single reference remains usable for non-diversity metrics (e.g. coherence, perplexity)

**Models<.**
| Key | Model |
|---|---|
| `llama` | meta-llama/Llama-3.1-8B-Instruct |
| `mistral` | mistralai/Mistral-7B-Instruct-v0.3 |
| `qwen` | Qwen/Qwen2.5-7B-Instruct |
| `gemma` | google/gemma-2-9b-it |

All four are open-weight models run locally via vLLM

**Setup:**
- GPU: a L4 or A100 is recommended
- Run one model per session
- vLLM does not release GPU memory reliably, so after each model restart the runtime and change `MODEL_KEY`.

**Decoding configuration:**

Fixed across all models (identical to WritingPrompts):
- `temperature=1.0`
- `top_p=0.9`
- `repetition_penalty=1.0` (disabled)
- `n=5`.

`max_new_tokens`: summaries must reach >= 10 sentences

No `no_repeat_ngram` constraint is used (would manipulate lexical diversity)

## Installation

In [ ]:
# Restart the runtime after this cell
!pip install -q -U vllm datasets nltk
!pip install -q --upgrade huggingface_hub
!pip uninstall -y -q torchaudio torchvision

In [ ]:
import json
import pathlib
import random
import re
import time
from google.colab import drive


# Mount Drive
drive.mount('/content/drive')
OUT_DIR = pathlib.Path('/content/drive/MyDrive/Masterarbeit/generations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT_DIR)

## Configuration

In [ ]:
# Switch this between runs: 'llama' | 'mistral' | 'qwen' | 'gemma'
MODEL_KEY = 'mistral'


MODELS = {
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'gemma':   'google/gemma-2-9b-it',
}

MODEL_ID = MODELS[MODEL_KEY]

# Sampling
N_PROMPTS = 100          # number of articles drawn from the test split (parity with WritingPrompts)
N_SAMPLES = 5             # generations per article (n=5)
SEED = 42                 # controls both article sampling and generation

# Reference-summary length filter: 
# articles are only eligible for sampling if their human summary already reaches this many sentences
MIN_SUMMARY_SENTENCES = 10


# Decoding (identical to WritingPrompts, except MAX_NEW_TOKENS: summaries here target >= 10 sentences)
TEMPERATURE = 1.0
TOP_P = 0.9
MAX_NEW_TOKENS = 640       # enough headroom for a 10+ sentence summary (~350-450 words)
REPETITION_PENALTY = 1.0  # deliberately disabled

# Shared vLLM context budget across all 4 models. Kept low enough to fit
# gemma-2-9b-it's native 8192-token limit with margin; 
# Source documents are truncated to SRC_MAX_TOKENS (see Load Multi-News section) to leave room for
# the prompt template and MAX_NEW_TOKENS within this budget.
MAX_MODEL_LEN = 6144
SRC_MAX_TOKENS = MAX_MODEL_LEN - MAX_NEW_TOKENS - 300  # 300-token margin for system/user template text


random.seed(SEED)

print(f'Model:  {MODEL_ID}')

In [ ]:
# Llama and Gemma are gated (accept the license on each model page and store
# token in a .env file on Drive):
#   /content/drive/MyDrive/Masterarbeit/.env
# containing a single line:
#   HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx

if MODEL_KEY in ('llama', 'gemma'):
    from huggingface_hub import login

    env_path = OUT_DIR.parent / '.env'

    # Read Hugging Face token from .env file on Drive
    hf_token = dict(
        line.strip().split('=', 1)
        for line in env_path.read_text().splitlines()
        if line.strip()
    )['HF_TOKEN']

    login(token=hf_token)
    print('Logged in to Hugging Face')

## Load Multi-News

- Multi-News (Fabbri et al., 2019) pairs clusters of 2-10 news articles about
  the same event with a single professionally written digest summary
- `alexfabbri/multi_news` on Hugging Face
- Fields `document` articles joined by `" ||||| "` — and `summary`).
- The `summary` field is kept alongside each article as the single human reference

In [ ]:
from datasets import load_dataset
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

HF_REPO = 'alexfabbri/multi_news'

# Load the Multi-News test split

ds = load_dataset(HF_REPO, split='test', revision='refs/convert/parquet')
print('Columns:', ds.column_names)


def split_documents(raw_document):
    """Multi-News joins the cluster's articles with ' ||||| '
    Split and label them for the prompt."""
    parts = [p.strip() for p in raw_document.split('|||||') if p.strip()]
    return [re.sub(r'^NEWLINE_CHAR\s*', '', p).replace('NEWLINE_CHAR', '\n') for p in parts]


def count_sentences(text):
    """Sentence count via NLTK Punkt, matching metrics_lib.narrative's tokenizer
    (same >5-char filter as narrative.get_sentiment_arc) so a text that passes
    this filter also passes D_narr's MIN_SENTENCES filter downstream."""
    return len([s for s in sent_tokenize(text) if len(s.strip()) > 5])


records = [
    {
        'documents': split_documents(r['document']),
        'human_summary': r['summary'].strip(),
    }
    for r in ds
]

print(f'Loaded {len(records)} article clusters from the test split')

In [ ]:
# Filter to clusters whose human reference already reaches MIN_SUMMARY_SENTENCES

eligible = [r for r in records if count_sentences(r['human_summary']) >= MIN_SUMMARY_SENTENCES]

print(f'{len(eligible)} of {len(records)} clusters have a human summary >= {MIN_SUMMARY_SENTENCES} sentences')
print(f'({100 * len(eligible) / len(records):.0f}% pass rate)')

In [ ]:
# Draw the sample once and reuse it for all four models, so every model uses the same articles

sampled = random.Random(SEED).sample(eligible, N_PROMPTS)

for i, item in enumerate(sampled):
    item['prompt_id'] = f'mn_{i:04d}'
    item['human_summary_n_sentences'] = count_sentences(item['human_summary'])


# Save the sampled articles to a JSONL file for later use
with open(OUT_DIR / 'multinews_articles_sample.jsonl', 'w') as f:
    for item in sampled:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')


print(f'Saved {len(sampled)} article clusters')
print()
print('Example (first source article, truncated):')
print(sampled[0]['documents'][0][:300])
print()
print('Example human summary:')
print(sampled[0]['human_summary'][:400])

## Prompt template

- Identical structure across models (system + user turn, folded for Gemma)
- Explicit sentence-count instruction
- Source articles are labeled and separated so multi-document structure is visible to the model

In [ ]:
SYSTEM_PROMPT = 'You are a professional news editor contributing summaries to an online news digest.'

USER_TEMPLATE = (
    'Summarize the following news articles, which all cover the same story.\n\n'
    '{articles}\n\n'
    'Write a comprehensive summary of at least 10 sentences that synthesizes the '
    'key information across all the articles. Do not refer to the source '
    'articles. Write the summary as a self-contained news text. Output only '
    'the summary, with no title, no preamble and no commentary.'
)


def format_articles(article_list):
    return '\n\n'.join(f'Article {i+1}:\n{a}' for i, a in enumerate(article_list))


def build_messages(article_list):
    user_content = USER_TEMPLATE.format(articles=format_articles(article_list))

    # Gemma's chat template has no 'system' role (only 'user'/'model'), so the
    # system prompt is folded into the user turn instead.

    if MODEL_KEY == 'gemma':
        return [
            {'role': 'user', 'content': f'{SYSTEM_PROMPT}\n\n{user_content}'},
        ]

    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]

## Generation (vLLM)

Passing `n=5` in `SamplingParams` and not five separate requests

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Check installed versions of torch and vllm
!pip show torch vllm | grep -E "Name|Version"
!find / -name "libcudart.so*"

In [ ]:
# Load the CUDA runtime library with global symbol visibility so that vLLM can find it
import ctypes
ctypes.CDLL(
    '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libcudart.so.13',
    mode=ctypes.RTLD_GLOBAL,
)

import sys

# Jupyter/Colab's stdout and stderr (ipykernel.iostream.OutStream) do not
# implement fileno(), but vLLM calls sys.stdout.fileno() to briefly silence
# NCCL's native logging when it sets up the process group. Fake the real
# OS-level fd numbers (1 = stdout, 2 = stderr) so that call succeeds.
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

import glob, os

# Keep vLLM's engine in this process instead of a spawned subprocess. In a
# spawned child, ipykernel re-installs its own OutStream as sys.stdout, so
# the fileno() patch above (applied only in this process) would not carry
# over and the same crash would recur inside the child.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

nvidia_lib_dirs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(nvidia_lib_dirs)


from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def truncate_to_tokens(text, max_tokens):
    """Truncate text to at most max_tokens tokens under this model's tokenizer."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens])


def truncate_cluster(article_list, max_tokens):
    """Truncate an article cluster to a shared token budget, split evenly
    across its articles so no single long article crowds out the others."""
    per_article_budget = max(max_tokens // len(article_list), 50)
    return [truncate_to_tokens(a, per_article_budget) for a in article_list]


# Apply each model's own chat template and hand plain text to vLLM
text_prompts = [
    tokenizer.apply_chat_template(
        build_messages(truncate_cluster(item['documents'], SRC_MAX_TOKENS)),
        tokenize=False,
        add_generation_prompt=True,
    )
    for item in sampled
]

print(f'Prepared {len(text_prompts)} prompts')
print(f'Source token budget per cluster: {SRC_MAX_TOKENS} (model context {MAX_MODEL_LEN})')

In [ ]:
# Install torchvision for vLLM's image generation support (even if not used here)
!pip install -q torchvision --index-url https://download.pytorch.org/whl/cu128 --no-deps

In [ ]:
# Clear vLLM's model registry cache to ensure that the latest model classes are used
from vllm.model_executor.models import registry
registry._try_inspect_model_cls.cache_clear()

In [ ]:
# Initialize the vLLM model: use bfloat16 for speed and memory efficiency, and
# limit GPU memory utilization to 90%
# max_model_len is shared across all 4 models (see Configuration section) so
# that source-article truncation is identical regardless of a model's native
# context length

llm = LLM(
    model=MODEL_ID,
    dtype='bfloat16',
    gpu_memory_utilization=0.90,
    max_model_len=MAX_MODEL_LEN,
    seed=SEED,
)

# Set up the sampling parameters for text generation
sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
    seed=SEED,
)

# Generate summaries for the article clusters using vLLM
start = time.time()
outputs = llm.generate(text_prompts, sampling_params)

print(f'Generation took {time.time() - start:.1f}s')

In [ ]:
results = []

# Collect the generated summaries and their metadata into a list of dictionaries
for item, output in zip(sampled, outputs):
    for k, candidate in enumerate(output.outputs):
        summary = candidate.text.strip()
        results.append({
            'prompt_id': item['prompt_id'],
            'model': MODEL_ID,
            'sample_idx': k,
            'summary': summary,
            'n_sentences': count_sentences(summary),
            'finish_reason': candidate.finish_reason,
            'n_tokens': len(candidate.token_ids),
            'source_documents': item['documents'],
            'human_summary': item['human_summary'],
        })

print(f'Collected {len(results)} summaries')

In [ ]:
# Language drift check: flag summaries that contain non-Latin script (CJK = Chinese, Japanese, Korean)

CJK = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]')

for record in results:
    hits = CJK.findall(record['summary'])
    record['cjk_chars'] = len(hits)
    record['language_ok'] = len(hits) == 0

flagged = [r for r in results if not r['language_ok']]

print(f'Summaries with non-Latin script: {len(flagged)} of {len(results)}')

for record in flagged:
    share = record['cjk_chars'] / len(record['summary'])
    print(f"  {record['prompt_id']}/{record['sample_idx']}  {share:.0%} CJK")

## Save results

In [ ]:
out_path = OUT_DIR / f'multinews_{MODEL_KEY}_n{N_SAMPLES}.jsonl'

# Save the generated summaries to a JSONL file
with open(out_path, 'w') as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


# Store the configuration (own filename, so it does not collide with the
# WritingPrompts/SummEval config_{MODEL_KEY}.json written to the same OUT_DIR)
config = {
    'model_id': MODEL_ID,
    'dataset': 'Multi-News (Fabbri et al., 2019), test split via alexfabbri/multi_news',
    'n_articles': len(sampled),
    'n_samples': N_SAMPLES,
    'seed': SEED,
    'min_summary_sentences': MIN_SUMMARY_SENTENCES,
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'max_new_tokens': MAX_NEW_TOKENS,
    'max_model_len': MAX_MODEL_LEN,
    'src_max_tokens': SRC_MAX_TOKENS,
    'repetition_penalty': REPETITION_PENALTY,
    'system_prompt': SYSTEM_PROMPT,
    'user_template': USER_TEMPLATE,
    'note': 'Only one human reference per article (dataset limitation); no human-vs-human diversity comparison for this task.',
}

with open(OUT_DIR / f'multinews_config_{MODEL_KEY}.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)


print('Saved:', out_path)

## Sanity checks

- `finish_reason == 'length'`: summary was cut off mid-sentence
-> raise `MAX_NEW_TOKENS` or trim to the last complete sentence
- `n_sentences < MIN_SUMMARY_SENTENCES`
- Length distribution per model: systematic differences
- Preambles: like "Here is a summary:" appear despite the instruction

In [ ]:
import collections
import statistics as st

# Compute and print summary statistics for the generated summaries
lengths = [len(record['summary'].split()) for record in results]
truncated = sum(1 for record in results if record['finish_reason'] == 'length')

print(f'Summaries:    {len(results)}')
print(f'Median words: {st.median(lengths):.0f}')
print(f'Mean words:   {st.mean(lengths):.0f} (SD {st.stdev(lengths):.0f})')
print(f'Min / max:    {min(lengths)} / {max(lengths)}')
print(f'Truncated:    {truncated} ({100 * truncated / len(results):.1f}%)')

In [ ]:
# Share of generations that actually reach MIN_SUMMARY_SENTENCES

sentence_counts = [record['n_sentences'] for record in results]
below_floor = sum(1 for n in sentence_counts if n < MIN_SUMMARY_SENTENCES)

print(f'Median sentences: {st.median(sentence_counts):.0f}')
print(f'Mean sentences:   {st.mean(sentence_counts):.1f} (SD {st.stdev(sentence_counts):.1f})')
print(f'Below {MIN_SUMMARY_SENTENCES}-sentence floor: {below_floor} ({100 * below_floor / len(results):.1f}%)')

# Per-prompt: how many of the 5 samples survive the floor (need >= 2 for any pairwise diversity metric)
by_prompt = collections.defaultdict(list)
for r in results:
    by_prompt[r['prompt_id']].append(r['n_sentences'] >= MIN_SUMMARY_SENTENCES)

survivors = [sum(v) for v in by_prompt.values()]
under_2 = sum(1 for s in survivors if s < 2)
print(f'Prompts with < 2 surviving generations (diversity metrics undefined): {under_2} of {len(survivors)}')

In [ ]:
# Look for repeated openings

openings = collections.Counter(' '.join(r['summary'].split()[:3]) for r in results)

print('Most frequent openings:')
for opening, count in openings.most_common(5):
    print(f'  {count:3d}x  {opening}')

print()
print('--- Sample summary ---')
print(results[0]['summary'][:600])